In [1]:
import argparse
from itertools import product

import numpy as np

from cc2cc import add_args, cc, ucc
from cc2cc.utils import Grid, gen_mole, print_computer_info
from cc2cc.utils.env_var import DATA_PATH
from cc2cc.utils.parser import gen_name_args

origin_mol_str_list = [
    # "molecule1-W4_11",
    "molecule2-W4_11",
    # "molecule3-W4_11",
    # "molecule4-W4_11",
    # "molecule5-W4_11",
]
name_mol_str_list = [
    # "molecule1",
    "molecule2",
    # "molecule3",
    # "molecule4",
    # "molecule5",
    # "molecule6",
]
name_mol_str_exclude_list = [
    "W4_11-propane",  # 3
    "molecule1-W4_11",
    "molecule2-W4_11",
    "molecule3-W4_11",
    "molecule4-W4_11",
    "molecule5-W4_11",
    "molecule1-ACC24",
    "molecule1-GAPS",
    "molecule1-GW100",
    "molecule1-MRADC",
    "molecule1-S30L",
    "molecule2-ACC24",
    "molecule2-GAPS",
    "molecule2-GW100",
    "molecule2-MRADC",
    "molecule2-S30L",
    "molecule3-ACC24",
    "molecule3-GAPS",
    "molecule3-GW100",
    "molecule3-MRADC",
    "molecule3-S30L",
    "molecule4-ACC24",
    "molecule4-GAPS",
    "molecule4-GW100",
    "molecule4-MRADC",
    "molecule4-S30L",
    "molecule5-ACC24",
    "molecule5-GAPS",
    "molecule5-GW100",
    "molecule5-MRADC",
    "molecule5-S30L",
    "molecule6-ACC24",
    "molecule6-GAPS",
    "molecule6-GW100",
    "molecule6-MRADC",
    "molecule6-S30L",
    "molecule7-ACC24",
    "molecule7-GAPS",
    "molecule7-GW100",
    "molecule7-MRADC",
    "molecule7-S30L",
]

mol_elements_dict = {}
len_elements_dict = {}

if __name__ == "__main__":
    error_molecule = []

    name_mol_list = gen_name_args(name_mol_str_list, "gmtkn-def2")
    name_mol_exclude_list = gen_name_args(
        name_mol_str_exclude_list, "gmtkn-def2", if_exclude=True
    )
    name_mol_list = [mol for mol in name_mol_list if mol not in name_mol_exclude_list]

    origin_mol_list = gen_name_args(origin_mol_str_list, "gmtkn-def2")
    name_mol_list = origin_mol_list + name_mol_list

    error_molecule = []
    print(f"Name Molecule List: {name_mol_list}")

    for name_mol in name_mol_list:
        try:
            mol = gen_mole(
                name_mol,
                0,
                1,
                0,
                "cc-pVDZ",
                "gmtkn-def2",
                if_rotate=True,
                if_rotate_random=False,
                solve_symmetry=True,
                verbose=1,
            )

            mol_elements = list(np.array(mol.elements))
            mol_atom_coords = list(mol.atom_coords())
            mol_atom_coords.append(name_mol)
            # print(mol_atom_coords)
            mol_elements.extend([mol.charge, mol.spin])
            mol_elements_str = "-".join(map(str, mol_elements))
            if mol_elements_str not in mol_elements_dict:
                mol_elements_dict[mol_elements_str] = [mol_atom_coords]
            else:
                mol_elements_dict[mol_elements_str].append(mol_atom_coords)
            len_elements_dict[mol_elements_str] = len(list(np.array(mol.elements)))

        except (ValueError, RuntimeError) as e:
            print(f"ERROR: {name_mol}")
            print(e)
            error_molecule.append(name_mol)
            print(f"Error molecule: {error_molecule}")
        finally:
            print(f"Processed: {name_mol}")
        print()

    print(f"Error molecule: {error_molecule}")

ModuleNotFoundError: No module named 'cc2cc'

In [2]:
for mol_elements_name, mol_elements in mol_elements_dict.items():
    # print(f"Processing {mol_elements_name}")
    # print(f"{mol_elements}")

    if (len_elements_dict[mol_elements_name] > 12):
        print("Skip large molecules (>12 atoms)\n")
        continue

    identifiables = [0]
    for i_elements in range(1, len(mol_elements)):
        distance_list = np.zeros(len(identifiables))
        for iter, identifiable in enumerate(identifiables):
            for i_element in range(len(mol_elements[i_elements]) - 1):
                distance_list[iter] = max(
                    np.linalg.norm(
                        mol_elements[i_elements][i_element]
                        - mol_elements[identifiable][i_element]
                    ),
                    distance_list[iter],
                )
        if np.all(distance_list > 0.1):
            identifiables.append(i_elements)
        # else:
        #     print(f"Skipping {mol_elements[i_elements][-1]}")

    # print("===identifiables===")
    for identifiable in identifiables:
        print(f'"{mol_elements[identifiable][-1]}",')
        # print(
        #     f"{np.array2string(np.array(mol_elements[identifiable][:-1]), formatter={'float': '{: .2f}'.format})}"
        # )
    print("")

NameError: name 'mol_elements_dict' is not defined

In [ ]:
len(origin_mol_list)

22